per neuron alignment

In [252]:
import torch
import pickle
import numpy
import re
import pandas
import os

def  get_indiv_concepts(formula) -> set:
    concepts = set()
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        concepts.add(c[:end_idx])
    return concepts
import pickle
with open("/workspace/CCE_NLI/code/Abstractions/final_abstractions.pkl", 'rb') as f:
    abs_map = pickle.load(f)

def find_cluster(raw_concept):
    for cluster in abs_map:
        if raw_concept in abs_map[cluster]:
            return cluster
def find_abstractions(expls):
    if isinstance(expls, set):
        glob = list(expls)
    else:
        glob=expls
    abstracts=[]
    exact_concepts_perabs=defaultdict(list)
    for concept in glob:
        raw_concept = concept.split(":")[-1]
        abstraction_cluster = find_cluster(raw_concept)
        if abstraction_cluster==143: 
            continue
        if not abstraction_cluster:abstraction_cluster=150
        exact_concepts_perabs[abstraction_cluster].append(raw_concept)
        
        abstracts.append(abstraction_cluster)
    for a in exact_concepts_perabs:
        exact_concepts_perabs[a] = Counter(exact_concepts_perabs[a])
    return Counter(abstracts), set(abstracts), dict(
                                                    sorted(
                                                        exact_concepts_perabs.items(),
                                                        key=lambda x: sum(x[1].values()),
                                                        reverse=True
                                                    )
)


def get_groups(formula):
    indiv = get_indiv_concepts(formula)
    groups = []
    for length in range(2, 3):
        for combo in combinations(indiv, length):
            groups.append(tuple(sorted(combo)))
    return sorted(groups)

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    raw=[]
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
        raw.extend(concepts)
    
    return unit_concepts, set(raw)

def build_binary_mask(neuron_mask, foundational_concept_list) -> torch.Tensor:
    num_neurons = len(neuron_mask)
    num_concepts = len(foundational_concept_list)

    # Step 1: Initialize tensor
    tensor = torch.zeros((num_neurons, num_concepts), dtype=torch.float32)

    # Step 2: Fill in ones
    for i, concepts in enumerate(neuron_mask.values()):
        for j, concept in enumerate(foundational_concept_list):
            if concept in concepts:
                tensor[i, j] = 1.0

    # Step 3: Compute row sums
    row_sums = tensor.sum(dim=1, keepdim=True)

    # Step 4: Normalize safely
    dist_tensor = torch.zeros_like(tensor)
    row_mask = (row_sums != 0).squeeze(1)  # True for rows with sum > 0
    dist_tensor[row_mask] = tensor[row_mask] #/ row_sums[row_mask]

    return dist_tensor



def get_neurons_for_cps(concepts, mapping):
    neurons = []
    for neuron, cps in mapping.items():
        for c in cps:
            if c in concepts:
                neurons.append(neuron)
                break
    return neurons

def get_all_cps_for_pi(folder):
    root_path = Path(folder)

    # Find all matching CSV files
    csv_pattern = 'Cluster*IOUS1024N.csv'
    csv_files = list(root_path.rglob(csv_pattern))
    
    concept_dict=defaultdict(lambda: defaultdict(set))
    s=set()
    for csv_file in csv_files:
        concepts = []
        csv_file = os.path.join(folder, csv_file)
        df = pd.read_csv(csv_file)
        for unit, formula in zip(df.unit, df.best_name):
            concept_dict[csv_file.split("/")[-2]][csv_file.split("/")[-1]].update(get_indiv_concepts(formula))
            s.update(get_indiv_concepts(formula))
     
      
    return concept_dict, s


In [215]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def all_correct_to_wrong(dense_cw, sparse_cw):
  
    
    return set(dense_cw['correct']) & set(sparse_cw['wrong'])
        

def find_highest_activating_neuron(samples_activations,model_finallayerweights, d):
    num_activ = (samples_activations>0).sum()
    flweights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    flweights=flweights.reshape((3,flweights.shape[0]//3 ))
    contribution = torch.tensor(samples_activations).abs().squeeze(0) *flweights
    
    total = contribution.sum(dim=1).argmax()   # [1024]

    most_impactful = contribution[total].argmax().item()
    
    return [most_impactful]
    

def find_highest_activating_neuron_at_cluster(cluster_mask, samples_activations,model_finallayerweights, d):
    num_activ = (samples_activations>0).sum()
   
    
    fl_cluster_weights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    
    fl_cluster_weights=fl_cluster_weights.reshape((3,fl_cluster_weights.shape[0]//3 ))
  
    contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights
    
    total = contribution.sum(dim=1).argmax()   # [1024]

    most_impactful = contribution[total].argmax().item()
    
    return [most_impactful]
    
def activationsiou(a,b):
    a = torch.where(a>0, 1,0)
    b = torch.where(b>0, 1,0)
    return (a&b).sum() / (a|b).sum() 
    
def iou(a,b):
    if isinstance(a, torch.Tensor):
        return (a&b).sum() / (a|b).sum()
    return len(a&b) / len(a|b) if len(a|b) > 0 else 0
ignore=0
def percent_sparse_in_dense(s,d):
    if len(d)==0 or len(s)==0: 
        return 1
    if len(d)==0 or len(s)==0: return 0
    return len(s&d)/len(d)

def concept_diff(expl_dense, expl_sparse):
    overlap = iou(set(expl_dense), set(expl_sparse))
    return 1-overlap

def get_mask(root,  cluster):
    return torch.load(os.path.join(root, f'Cluster{cluster}masks.pt'), map_location=device).t()

def get_subactiv(root, cluster):
    mask = get_mask(root,  cluster)
    dead=[]
    for n,i in enumerate(mask):
        if i.sum() < 500:
            dead.append(n)
    return dead

concept alignment (clusterwise per neuron and holistiic, and across cluster per neuron and holisic

In [253]:
model = 'LLAMA'
method = 'lottery_ticket'
c = 1

print(f"{model} {method} CLUSTER {c}")

# ── paths ─────────────────────────────────────────────────────────────────────
path_to_experiment = os.path.join("/workspace/CCE_NLI", model, 'exp', method, 'Run0.25_5')
dense_path_root    = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
sparse_path_root   = path_to_experiment

# ── dense activations ─────────────────────────────────────────────────────────
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────────
dense_expls1, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster1IOUS1024N.csv'))
dense_expls2, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster2IOUS1024N.csv'))
dense_expls3, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster3IOUS1024N.csv'))

# ── dense masks ───────────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_path_root, 'Masks', '0.0%Pruned')
dense_mask1 = get_mask(dense_mask_path, cluster=1).t()
dense_mask2 = get_mask(dense_mask_path, cluster=2).t()
dense_mask3 = get_mask(dense_mask_path, cluster=3).t()
dense_dead  = get_subactiv(dense_mask_path, cluster=c)
print(f"Dense masks loaded — shapes: {dense_mask1.shape}, {dense_mask2.shape}, {dense_mask3.shape}")

# ── dense model weights ───────────────────────────────────────────────────────
dense_model_finallayerweights = torch.load(
    f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth',
    map_location=device
)['state_dict']['mlp.3.weight']

# ── sparsity loop ─────────────────────────────────────────────────────────────
start = 1

clusters_dense = {
    1: (dense_mask1, dense_expls1),
    2: (dense_mask2, dense_expls2),
    3: (dense_mask3, dense_expls3),
}

start = 1

for i, sparsity in enumerate(sorted(os.listdir(os.path.join(path_to_experiment, 'Expls')))):

    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity:
        continue

    print(f"\n── {sparsity} (Pruning iter {start}) ──")

    # ── sparse activations ─────────────────────────────────────────
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))

    # ── sparse explanations ────────────────────────────────────────
    sparse_expls1, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster1IOUS1024N.csv'))
    sparse_expls2, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster2IOUS1024N.csv'))
    sparse_expls3, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster3IOUS1024N.csv'))

    clusters_sparse = {
        1: sparse_expls1,
        2: sparse_expls2,
        3: sparse_expls3,
    }

    # ── sparse masks ───────────────────────────────────────────────
    sparse_mask_path = os.path.join(sparse_path_root, 'Masks', sparsity)
    sparse_mask1 = get_mask(sparse_mask_path, cluster=1).t()
    sparse_mask2 = get_mask(sparse_mask_path, cluster=2).t()
    sparse_mask3 = get_mask(sparse_mask_path, cluster=3).t()
    sparse_dead  = get_subactiv(sparse_mask_path, cluster=c)

    clusters_sparse_masks = {
        1: sparse_mask1,
        2: sparse_mask2,
        3: sparse_mask3,
    }

    # ── sparse model weights ───────────────────────────────────────
    sparse_model_finallayerweights = torch.load(
        f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth',
        map_location=device
    )['state_dict']['mlp.3.weight']
    if method == 'CoFi':
        zs = torch.load(
            f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt',
            map_location=device
        )
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(
            zs['final_mlp_hidden_z'].to(device)
        )

    # ── metrics init ───────────────────────────────────────────────
    cluster_holistic = {1: 0, 2: 0, 3: 0}
    cluster_per_neuron = {1: 0, 2: 0, 3: 0}
    cluster_counts = {1: 0, 2: 0, 3: 0}

    concept_hol_dif_all = 0
    per_neuron_all = 0
    per_neuron_all_count = 0

    # ── main loop ─────────────────────────────────────────────────
    for sample in range(600):

        all_sparse_neurons = set()
        all_dense_neurons = set()

        sparse_neurons_dict = {}
        dense_neurons_dict = {}

        for cl in [1, 2, 3]:

            dense_mask, dense_expls = clusters_dense[cl]
            sparse_mask = clusters_sparse_masks[cl]
       
            sparse_expls = clusters_sparse[cl]
        
            sparse_neurons = find_highest_activating_neurons_at_cluster(
                sparse_mask.t()[sample],
                sparse_activations[sample],
                sparse_model_finallayerweights,
                sparse_dead
            )

            dense_neurons = find_highest_activating_neurons_at_cluster(
                dense_mask.t()[sample],
                dense_activations[sample],
                dense_model_finallayerweights,
                dense_dead
            )

            sparse_neurons_dict[cl] = sparse_neurons
            dense_neurons_dict[cl] = dense_neurons

            all_sparse_neurons |= set(sparse_neurons)
            all_dense_neurons |= set(dense_neurons)

            # ── per-cluster holistic ──
            if len(dense_neurons) > 0:
                s_union = set().union(*[sparse_expls[n] for n in sparse_neurons]) if sparse_neurons else set()
                d_union = set().union(*[dense_expls[n] for n in dense_neurons]) if dense_neurons else set()

                if len(d_union) > 0:
                    cluster_holistic[cl] += len(s_union & d_union) / len(s_union | d_union)

            # ── per-cluster per-neuron ──
            for s,d in zip(set(sparse_neurons), set(dense_neurons)):

                s_concepts = sparse_expls[s]
                d_concepts = dense_expls[d]

                if len(d_concepts) == 0:
                    continue

                overlap = len(s_concepts & d_concepts) / len(s_concepts | d_concepts)

                cluster_per_neuron[cl] += overlap
                cluster_counts[cl] += 1

        # ── all clusters holistic ──
        s_union_all = set()
        d_union_all = set()

        for cl in [1, 2, 3]:
            s_union_all |= set().union(*[clusters_sparse[cl][n] for n in sparse_neurons_dict[cl]]) if sparse_neurons_dict[cl] else set()
            d_union_all |= set().union(*[clusters_dense[cl][1][n] for n in dense_neurons_dict[cl]]) if dense_neurons_dict[cl] else set()

        if len(d_union_all) > 0:
            concept_hol_dif_all += len(s_union_all & d_union_all) / len(s_union_all|d_union_all)

        # ── all clusters per-neuron ──
        for s,d in zip(all_sparse_neurons, all_dense_neurons):

            s_concepts = set()
            d_concepts = set()

            for cl in [1, 2, 3]:
                if n in clusters_sparse[cl]:
                    s_concepts |= clusters_sparse[cl][s]
                if n in clusters_dense[cl][1]:
                    d_concepts |= clusters_dense[cl][1][d]

            if len(d_concepts) == 0:
                continue

            overlap = len(s_concepts & d_concepts) / len(s_concepts|d_concepts)

            per_neuron_all += overlap
            per_neuron_all_count += 1

    # ── normalize ─────────────────────────────────────────────────
    num_samples = 600

    cluster_holistic = {cl: cluster_holistic[cl] / num_samples for cl in cluster_holistic}

    cluster_per_neuron = {
        cl: (cluster_per_neuron[cl] / cluster_counts[cl] if cluster_counts[cl] > 0 else 0)
        for cl in cluster_per_neuron
    }

    concept_hol_dif_all /= num_samples
    per_neuron_all = per_neuron_all / per_neuron_all_count if per_neuron_all_count > 0 else 0

    # ── print results ─────────────────────────────────────────────
    print("Cluster Holistic:", cluster_holistic)
    print("Cluster Per-Neuron:", cluster_per_neuron)
    print("All Clusters Holistic:", concept_hol_dif_all)
    print("All Clusters Per-Neuron:", per_neuron_all)

    start += 1

LLAMA lottery_ticket CLUSTER 1
Dense masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])


KeyboardInterrupt: 

In [ ]:
concept preservatoin (clusterwise per neuron and holistiic, and across cluster per neuron and holisic

In [265]:
model = 'BOWMAN'
method = 'wanda'
c = 1

print(f"{model} {method} CLUSTER {c}")

# ── paths ─────────────────────────────────────────────────────────────────────
path_to_experiment = os.path.join("/workspace/CCE_NLI", model, 'exp', method, 'Run0.25_5')
dense_path_root    = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
sparse_path_root   = path_to_experiment

# ── dense activations ─────────────────────────────────────────────────────────
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────────
dense_expls1, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster1IOUS1024N.csv'))
dense_expls2, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster2IOUS1024N.csv'))
dense_expls3, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster3IOUS1024N.csv'))

# ── dense masks ───────────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_path_root, 'Masks', '0.0%Pruned')
dense_mask1 = get_mask(dense_mask_path, cluster=1).t()
dense_mask2 = get_mask(dense_mask_path, cluster=2).t()
dense_mask3 = get_mask(dense_mask_path, cluster=3).t()
dense_dead  = get_subactiv(dense_mask_path, cluster=c)
print(f"Dense masks loaded — shapes: {dense_mask1.shape}, {dense_mask2.shape}, {dense_mask3.shape}")

# ── dense model weights ───────────────────────────────────────────────────────
dense_model_finallayerweights = torch.load(
    f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth',
    map_location=device
)['state_dict']['mlp.3.weight']

# ── sparsity loop ─────────────────────────────────────────────────────────────
start = 1

clusters_dense = {
    1: (dense_mask1, dense_expls1),
    2: (dense_mask2, dense_expls2),
    3: (dense_mask3, dense_expls3),
}

start = 1

for i, sparsity in enumerate(sorted(os.listdir(os.path.join(path_to_experiment, 'Expls')))):

    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity:
        continue

    print(f"\n── {sparsity} (Pruning iter {start}) ──")

    # ── sparse activations ─────────────────────────────────────────
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))

    # ── sparse explanations ────────────────────────────────────────
    sparse_expls1, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster1IOUS1024N.csv'))
    sparse_expls2, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster2IOUS1024N.csv'))
    sparse_expls3, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster3IOUS1024N.csv'))

    clusters_sparse = {
        1: sparse_expls1,
        2: sparse_expls2,
        3: sparse_expls3,
    }

    # ── sparse masks ───────────────────────────────────────────────
    sparse_mask_path = os.path.join(sparse_path_root, 'Masks', sparsity)
    sparse_mask1 = get_mask(sparse_mask_path, cluster=1).t()
    sparse_mask2 = get_mask(sparse_mask_path, cluster=2).t()
    sparse_mask3 = get_mask(sparse_mask_path, cluster=3).t()
    sparse_dead  = get_subactiv(sparse_mask_path, cluster=c)

    clusters_sparse_masks = {
        1: sparse_mask1,
        2: sparse_mask2,
        3: sparse_mask3,
    }

    # ── sparse model weights ───────────────────────────────────────
    sparse_model_finallayerweights = torch.load(
        f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth',
        map_location=device
    )['state_dict']['mlp.3.weight']
    if method == 'CoFi':
        zs = torch.load(
            f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt',
            map_location=device
        )
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(
            zs['final_mlp_hidden_z'].to(device)
        )

    # ── metrics init ───────────────────────────────────────────────
    cluster_holistic = {1: 0, 2: 0, 3: 0}
    cluster_per_neuron = {1: 0, 2: 0, 3: 0}
    cluster_counts = {1: 0, 2: 0, 3: 0}

    concept_hol_dif_all = 0
    per_neuron_all = 0
    per_neuron_all_count = 0

    # ── main loop ─────────────────────────────────────────────────
    for sample in range(600):

        all_sparse_neurons = set()
        all_dense_neurons = set()

        sparse_neurons_dict = {}
        dense_neurons_dict = {}

        for cl in [1, 2, 3]:

            dense_mask, dense_expls = clusters_dense[cl]
            sparse_mask = clusters_sparse_masks[cl]
       
            sparse_expls = clusters_sparse[cl]
        
            sparse_neurons = find_highest_activating_neurons_at_cluster(
                sparse_mask.t()[sample],
                sparse_activations[sample],
                sparse_model_finallayerweights,
                sparse_dead
            )

            dense_neurons = find_highest_activating_neurons_at_cluster(
                dense_mask.t()[sample],
                dense_activations[sample],
                dense_model_finallayerweights,
                dense_dead
            )

            sparse_neurons_dict[cl] = sparse_neurons
            dense_neurons_dict[cl] = dense_neurons

            all_sparse_neurons |= set(sparse_neurons)
            all_dense_neurons |= set(dense_neurons)

            # ── per-cluster holistic ──
            if len(dense_neurons) > 0:
                s_union = set().union(*[sparse_expls[n] for n in sparse_neurons]) if sparse_neurons else set()
                d_union = set().union(*[dense_expls[n] for n in dense_neurons]) if dense_neurons else set()

                if len(d_union) > 0:
                    cluster_holistic[cl] += len(s_union & d_union) / len(d_union)

            # ── per-cluster per-neuron ──
            for s,d in zip(set(sparse_neurons), set(dense_neurons)):

                s_concepts = sparse_expls[s]
                d_concepts = dense_expls[d]

                if len(d_concepts) == 0:
                    continue

                overlap = len(s_concepts & d_concepts) / len( d_concepts)

                cluster_per_neuron[cl] += overlap
                cluster_counts[cl] += 1

        # ── all clusters holistic ──
        s_union_all = set()
        d_union_all = set()

        for cl in [1, 2, 3]:
            s_union_all |= set().union(*[clusters_sparse[cl][n] for n in sparse_neurons_dict[cl]]) if sparse_neurons_dict[cl] else set()
            d_union_all |= set().union(*[clusters_dense[cl][1][n] for n in dense_neurons_dict[cl]]) if dense_neurons_dict[cl] else set()

        if len(d_union_all) > 0:
            concept_hol_dif_all += len(s_union_all & d_union_all) / len(d_union_all)

        # ── all clusters per-neuron ──
        for s,d in zip(all_sparse_neurons, all_dense_neurons):

            s_concepts = set()
            d_concepts = set()

            for cl in [1, 2, 3]:
                if n in clusters_sparse[cl]:
                    s_concepts |= clusters_sparse[cl][s]
                if n in clusters_dense[cl][1]:
                    d_concepts |= clusters_dense[cl][1][d]

            if len(d_concepts) == 0:
                continue

            overlap = len(s_concepts & d_concepts) / len(d_concepts)

            per_neuron_all += overlap
            per_neuron_all_count += 1

    # ── normalize ─────────────────────────────────────────────────
    num_samples = 600

    cluster_holistic = {cl: cluster_holistic[cl] / num_samples for cl in cluster_holistic}

    cluster_per_neuron = {
        cl: (cluster_per_neuron[cl] / cluster_counts[cl] if cluster_counts[cl] > 0 else 0)
        for cl in cluster_per_neuron
    }

    concept_hol_dif_all /= num_samples
    per_neuron_all = per_neuron_all / per_neuron_all_count if per_neuron_all_count > 0 else 0

    # ── print results ─────────────────────────────────────────────
    print("Cluster Holistic:", cluster_holistic)
    print("Cluster Per-Neuron:", cluster_per_neuron)
    print("All Clusters Holistic:", concept_hol_dif_all)
    print("All Clusters Per-Neuron:", per_neuron_all)

    start += 1

BOWMAN wanda CLUSTER 1
Dense masks loaded — shapes: torch.Size([1024, 10000]), torch.Size([1024, 10000]), torch.Size([1024, 10000])

── 25.0%Pruned (Pruning iter 1) ──


/tmp/ipykernel_350/211658876.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights


Cluster Holistic: {1: 0.4205260092576272, 2: 0.4368781572458042, 3: 0.08973611111111116}
Cluster Per-Neuron: {1: 0.2609260423213912, 2: 0.4997003227293687, 3: 0.547058823529412}
All Clusters Holistic: 0.6045180555897349
All Clusters Per-Neuron: 0

── 43.75%Pruned (Pruning iter 2) ──


/tmp/ipykernel_350/211658876.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights


Cluster Holistic: {1: 0.4255109910514329, 2: 0.3470436937789881, 3: 0.10133333333333328}
Cluster Per-Neuron: {1: 0.26007893139040644, 2: 0.32173086067522616, 3: 0.5401960784313725}
All Clusters Holistic: 0.5971345253150004
All Clusters Per-Neuron: 0

── 57.812%Pruned (Pruning iter 3) ──


...

Cluster Holistic: {1: 0.3804177766677767, 2: 0.30259766894325757, 3: 0.09329166666666669}
Cluster Per-Neuron: {1: 0.2610016420361247, 2: 0.24627534181989666, 3: 0.5124378109452735}
All Clusters Holistic: 0.526154658249686
All Clusters Per-Neuron: 0

── 68.359%Pruned (Pruning iter 4) ──


/tmp/ipykernel_350/211658876.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights


Cluster Holistic: {1: 0.36857396225778594, 2: 0.35946449519978957, 3: 0.09583333333333328}
Cluster Per-Neuron: {1: 0.20154926866820622, 2: 0.3345410628019326, 3: 0.6239583333333335}
All Clusters Holistic: 0.5113180369963315
All Clusters Per-Neuron: 0

── 76.27%Pruned (Pruning iter 5) ──


/tmp/ipykernel_350/211658876.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights


Cluster Holistic: {1: 0.4399768032893034, 2: 0.26543121992386703, 3: 0.09111111111111106}
Cluster Per-Neuron: {1: 0.190734649122807, 2: 0.1546969696969697, 3: 0.4492307692307691}
All Clusters Holistic: 0.5418074574349664
All Clusters Per-Neuron: 0


neuron alingment pariwise neruon and acors neurons

In [ ]:
model = 'BERT'
method = 'CoFi'
c = 1

print(f"{model} {method} CLUSTER {c}")

# ── paths ─────────────────────────────────────────────────────────────────────
path_to_experiment = os.path.join("/workspace/CCE_NLI", model, 'exp', method, 'Run0.25_6')
dense_path_root    = f"/workspace/CCE_NLI/{model}/exp/CoFi/Run0.25_6"
sparse_path_root   = path_to_experiment

# ── dense activations ─────────────────────────────────────────────────────────
denseactivs = f"/workspace/CCE_NLI/{model}/activations/CoFi/Run0.25_6/0_Pruning_Iter/final_layer_activations.pkl"
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))

# ── dense explanations ────────────────────────────────────────────────────────
dense_expls1, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster1IOUS1024N.csv'))
dense_expls2, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster2IOUS1024N.csv'))
dense_expls3, _ = load_csv_data(os.path.join(dense_path_root, 'Expls', '0.0%Pruned', f'Cluster3IOUS1024N.csv'))

# ── dense masks ───────────────────────────────────────────────────────────────
dense_mask_path = os.path.join(dense_path_root, 'Masks', '0.0%Pruned')
dense_mask1 = get_mask(dense_mask_path, cluster=1).t()
dense_mask2 = get_mask(dense_mask_path, cluster=2).t()
dense_mask3 = get_mask(dense_mask_path, cluster=3).t()
dense_dead  = get_subactiv(dense_mask_path, cluster=c)
print(f"Dense masks loaded — shapes: {dense_mask1.shape}, {dense_mask2.shape}, {dense_mask3.shape}")

# ── dense model weights ───────────────────────────────────────────────────────
dense_model_finallayerweights = torch.load(
    f'/workspace/CCE_NLI/{model}/models/CoFi/Run0.25_6/0_Pruning_Iter/model_best.pth',
    map_location=device
)['state_dict']['mlp.3.weight']

# ── sparsity loop ─────────────────────────────────────────────────────────────
start = 1

for i, sparsity in enumerate(sorted(os.listdir(os.path.join(path_to_experiment, 'Expls')))):

    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity:
        continue

    print(f"\n── {sparsity} (Pruning iter {start}) ──")

    # sparse activations
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_6/{start}_Pruning_Iter/final_layer_activations.pkl"
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))

    # sparse explanations
    sparse_expls1, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster1IOUS1024N.csv'))
    sparse_expls2, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster2IOUS1024N.csv'))
    sparse_expls3, _ = load_csv_data(os.path.join(sparse_path_root, 'Expls', sparsity, 'Cluster3IOUS1024N.csv'))

    # sparse masks
    sparse_mask_path = os.path.join(sparse_path_root, 'Masks', sparsity)
    sparse_mask1 = get_mask(sparse_mask_path, cluster=1).t()
    sparse_mask2 = get_mask(sparse_mask_path, cluster=2).t()
    sparse_mask3 = get_mask(sparse_mask_path, cluster=3).t()
    sparse_dead  = get_subactiv(sparse_mask_path, cluster=c)
    print(f"Sparse masks loaded — shapes: {sparse_mask1.shape}, {sparse_mask2.shape}, {sparse_mask3.shape}")

    # sparse model weights
    sparse_model_finallayerweights = torch.load(
        f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_6/{start}_Pruning_Iter/model_best.pth',
        map_location=device
    )['state_dict']['mlp.3.weight']

    if method == 'CoFi':
        zs = torch.load(
            f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_6/{start}_Pruning_Iter/zs.pt',
            map_location=device
        )
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(
            zs['final_mlp_hidden_z'].to(device)
        )


    # ── per-sample alignment ──────────────────────────────────────────────────
    pairwise_alignment  = {1: defaultdict(float), 2: defaultdict(float), 3: defaultdict(float)}
    holistic_alignment  = {1: defaultdict(float), 2: defaultdict(float), 3: defaultdict(float)}

    cluster_masks_sparse = {1: (sparse_mask1, sparse_highest_neurons1 if False else None),
                            2: (sparse_mask2, None),
                            3: (sparse_mask3, None)}
    cluster_masks_dense  = {1: (dense_mask1,  None),
                            2: (dense_mask2,  None),
                            3: (dense_mask3,  None)}

    for c2w_sample in range(600):

        # top neurons per cluster — sparse
        print(sparse_activations[c2w_sample].shape ,
                sparse_model_finallayerweights.shape)
        sparse_top = {
            1: find_highest_activating_neurons_at_cluster(
                sparse_mask1.t()[c2w_sample], sparse_activations[c2w_sample],
                sparse_model_finallayerweights, sparse_dead),
            2: find_highest_activating_neurons_at_cluster(
                sparse_mask2.t()[c2w_sample], sparse_activations[c2w_sample],
                sparse_model_finallayerweights, sparse_dead),
            3: find_highest_activating_neurons_at_cluster(
                sparse_mask3.t()[c2w_sample], sparse_activations[c2w_sample],
                sparse_model_finallayerweights, sparse_dead),
        }

        # top neurons per cluster — dense
        dense_top = {
            1: find_highest_activating_neurons_at_cluster(
                dense_mask1.t()[c2w_sample], dense_activations[c2w_sample],
                dense_model_finallayerweights, dense_dead),
            2: find_highest_activating_neurons_at_cluster(
                dense_mask2.t()[c2w_sample], dense_activations[c2w_sample],
                dense_model_finallayerweights, dense_dead),
            3: find_highest_activating_neurons_at_cluster(
                dense_mask3.t()[c2w_sample], dense_activations[c2w_sample],
                dense_model_finallayerweights, dense_dead),
        }

        sparse_masks = {1: sparse_mask1, 2: sparse_mask2, 3: sparse_mask3}
        dense_masks  = {1: dense_mask1,  2: dense_mask2,  3: dense_mask3}

        for cl in [1, 2, 3]:

            # 1. pairwise alignment
            pairwise_scores = [
                iou(sparse_masks[cl][sparse_n], dense_masks[cl][dense_n]).item()
                for sparse_n, dense_n in zip(sparse_top[cl], dense_top[cl])
            ]
            pairwise_alignment[cl][c2w_sample] = sum(pairwise_scores) / len(pairwise_scores) \
                if pairwise_scores else 0.0

            # 2. holistic alignment — union within this cluster
            sparse_union = None
            for n in sparse_top[cl]:
                sparse_union = sparse_masks[cl][n] if sparse_union is None \
                    else (sparse_union | sparse_masks[cl][n])

            dense_union = None
            for n in dense_top[cl]:
                dense_union = dense_masks[cl][n] if dense_union is None \
                    else (dense_union | dense_masks[cl][n])

            holistic_alignment[cl][c2w_sample] = iou(sparse_union, dense_union).item() \
                if (sparse_union is not None and dense_union is not None) else 0.0

    # ── summary for this sparsity level ──────────────────────────────────────
    print(f"\n[{sparsity}]")
    for cl in [1, 2, 3]:
        avg_pair     = sum(pairwise_alignment[cl].values())  / len(pairwise_alignment[cl])
        avg_holistic = sum(holistic_alignment[cl].values())  / len(holistic_alignment[cl])
        print(f"  Cluster {cl} — Pairwise: {avg_pair:.4f}   Holistic: {avg_holistic:.4f}")

    start += 1

In [270]:
'''All c->w samples
Get highest activating neurons
Open mask for respective nerun
Calc alignment
Open expls for respective neuron
Clac concept dif %
At end: avg alignment and average concept dif %
If dif is high (close to 1) -> dif concepts entirely 
Alignment high: neurons behave the same so theyre learning dif ways to represent the same set → but those details are the wrong way
Alignment low: neurons explain vastly dif concepts and fire for very dif samples so the behavior of the neuron changes fully → wrong behavior
If dif is low (less than 50) -> dif combos 
High align: same behsvior even when misclassified
Low: wrong combos!'''

model='BERT'
method='lottery_ticket'
c=1

print(f"{model} {method} CLUSTER {c}")
path_to_experiment=os.path.join("/workspace/CCE_NLI",model, 'exp', method, 'Run0.25_5')
#dense_cw = pd.read_csv(os.path.join(path_to_experiment,  f'Prediction_CW_0_Pruning_Iter.csv')).set_index('Unnamed: 0').T
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
dense_expls1,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster{1}IOUS1024N.csv'))
dense_expls2,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster{2}IOUS1024N.csv'))
dense_expls3,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster{3}IOUS1024N.csv'))
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))
    
    
dense_path_root = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
dense_mask1 = get_mask(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=1)
dense_mask2 = get_mask(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=2)
dense_mask3 = get_mask(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=3)
start=1
dense_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
dense_dead=get_subactiv(os.path.join(dense_path_root, 'Masks', '0.0%Pruned'), cluster=c)
if '0.0%Pruned' in os.listdir(f"{sparse_path_root}/Expls"):
    start=1
    
sparse_path_root=f"{path_to_experiment}"
for i, sparsity in enumerate(sorted(os.listdir(f"{path_to_experiment}/Expls"))):

    print(sparse_path_root, sparsity)
    #if sparsity != '25.0%Pruned': continue
    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity: continue
    #sparse_cw = pd.read_csv(os.path.join(path_to_experiment, f'Prediction_CW_{start}_Pruning_Iter.csv')).set_index('Unnamed: 0').T
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    sparse_expls_1, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{1}IOUS1024N.csv'))
    sparse_expls_2, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{2}IOUS1024N.csv'))
    sparse_expls_3, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{3}IOUS1024N.csv'))
    
  
    
    sparse_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
    
    if method=='CoFi':
        zs=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt', map_location=device)
     
        print(zs['final_mlp_hidden_z'].device, sparse_model_finallayerweights.device)
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(zs['final_mlp_hidden_z'].to(device))
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))
        
    #c_2_w = all_correct_to_wrong(dense_cw, sparse_cw)
    
    per_neuron_average_alignment = {'1':defaultdict(list), '2':defaultdict(list), '3':defaultdict(list)}
    avg_cluster_holistic_alignment = 0
    
    alignment_between_neurons= []
    average_concept_diff = defaultdict(int)
    sparse_mask = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=c)
    print(f"LOADED FROM FILE SHPAE {sparse_mask.shape}")
    sparse_dead = get_subactiv(os.path.join(dense_path_root, 'Masks', sparsity), cluster=c)
    start += 1
    #print(sparsity, len(c_2_w))
    died_neurons=0
    revived_neurons=0
    dense_highest_neuronsset=set()
    sparse_mask_1 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=1)
    sparse_mask_2 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=2)
    sparse_mask_3 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=3)
       
        
    dense_highest_neurons1 = find_highest_activating_neurons_at_cluster(dense_mask_1[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
    dense_highest_neurons1 =set(dense_highest_neurons1)
    dense_highest_neurons2 = find_highest_activating_neurons_at_cluster(dense_mask_2[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
    dense_highest_neurons2=set(dense_highest_neurons2)
    dense_highest_neurons3 = find_highest_activating_neurons_at_cluster(dense_mask_3[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
    dense_highest_neurons3=set(dense_highest_neurons3)
        
    for c2w_sample in range(300): #[:100]:
 

   
        difference_in_concepts=0
        ignore = 0
       
        sparse_highest_neurons1 = find_highest_activating_neurons_at_cluster(sparse_mask_1[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons1=set(sparse_highest_neurons1)
        sparse_highest_neurons2= find_highest_activating_neurons_at_cluster(sparse_mask_2[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons2=set(sparse_highest_neurons2)
        sparse_highest_neurons3= find_highest_activating_neurons_at_cluster(sparse_mask_3[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons3 = set(sparse_highest_neurons3)
        holisitc_sparse = sparse_highest_neurons1.union(sparse_highest_neurons2).union(sparse_highest_neurons3)

        
        hol_sparse_highest_neurons = find_highest_activating_neurons( sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        hol_dense_highest_neurons = find_highest_activating_neurons(dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        
        for sparse_highest_neuron, dense_highest_neuron in zip(sparse_highest_neurons3, dense_highest_neurons3):  
            per_neuron_alignment=iou(sparse_mask_3[sparse_highest_neuron], dense_mask3[dense_highest_neuron])
            per_neuron_average_alignment['3'][c2w_sample].append(per_neuron_alignment.item())
            
        for sparse_highest_neuron, dense_highest_neuron in zip(sparse_highest_neurons2, dense_highest_neurons2):  
            per_neuron_alignment=iou(sparse_mask_2[sparse_highest_neuron], dense_mask2[dense_highest_neuron])
            per_neuron_average_alignment['2'][c2w_sample].append(per_neuron_alignment.item())
            
        for sparse_highest_neuron, dense_highest_neuron in zip(sparse_highest_neurons1, dense_highest_neurons1):  
            per_neuron_alignment=iou(sparse_mask_1[sparse_highest_neuron], dense_mask1[dense_highest_neuron])
            per_neuron_average_alignment['1'][c2w_sample].append(per_neuron_alignment.item())
            
        holisitc_sparse=list(holisitc_sparse)
        holisitc_dense=list(holisitc_dense)
        
        sparse_union_mask = sparse_activations.t()[holisitc_sparse].any(dim=0)
        dense_union_mask = dense_activations.t()[holisitc_dense].any(dim=0)
        avg_cluster_holistic_alignment  += activationsiou(sparse_union_mask, dense_union_mask)
            
           
            
        for cluster in per_neuron_average_alignment.keys():
            per_neuron_average_alignment[cluster][c2w_sample] = sum(per_neuron_average_alignment[cluster][c2w_sample])/len(per_neuron_average_alignment[cluster][c2w_sample])
    for cluster in per_neuron_average_alignment.keys():
        per_neuron_average_alignment[cluster]= sum(per_neuron_average_alignment[cluster].values())/len(per_neuron_average_alignment[cluster])
        
    print(f"per neuron Average alignment: ", per_neuron_average_alignment)
    print(f"Holistic average alignment: ", avg_cluster_holistic_alignment / 300)
        
   #for sample 8204 720 is the highst contributiig dense neuron and 718 is the highest contributiig sparse neruon (25%)  so 
        #see where these samples differ in the firing. (collect all the sentences into a txt file (2 sep ) and compare them see if any patterns)

BERT lottery_ticket CLUSTER 1
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5 0.0%Pruned
/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_5 25.0%Pruned
LOADED FROM FILE SHPAE torch.Size([10000, 1024])


/tmp/ipykernel_350/211658876.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights
/tmp/ipykernel_350/211658876.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights


KeyboardInterrupt: 

In [ ]:
no dif between preservation: neurons behave dif at all sparsities regardless of how pred changed
bert lth c->c
25.0%Pruned 8290
Average concept difference :  0.6654999999999999
Average alignment:  0.23897429130971432
    
43.75%Pruned 7811
Average concept difference :  0.6144999999999998
Average alignment:  0.17982538217678667

57.812%Pruned 6156
Average concept difference :  0.6798333333333332
Average alignment:  0.17942780851386486
    
68.359%Pruned 3780
Average concept difference :  0.7386666666666666
Average alignment:  0.1512690109340474
    
76.27%Pruned 2600
Average concept difference :  0.6961666666666667
Average alignment:  0.1716563373338431

    
bert lth c->w
25.0%Pruned 136
Average concept difference :  0.6161666666666665
Average alignment:  0.23048584141070022

43.75%Pruned 615
Average concept difference :  0.6861666666666666
Average alignment:  0.21354142345953733
    
57.812%Pruned 2270
Average concept difference :  0.7136666666666664
Average alignment:  0.1814640353480354
    
68.359%Pruned 4646
Average concept difference :  0.7028333333333333
Average alignment:  0.1619819349143654
    
76.27%Pruned 5826
Average concept difference :  0.6745
Average alignment:  0.16334903969895095


In [16]:
bert cofi c->w
0.25267370011269075%
Average concept difference :  54.25
Average alignment:  0.39514948163181546

0.4443280775114645%
Average concept difference :  59.28333333333332
Average alignment:  0.3604531835205853

0.5826748728764346%
Average concept difference :  46.649999999999997
Average alignment:  0.402138639902696

0.6841850626856109%Pruned 583
Average concept difference :  52.63333333333334
Average alignment:  0.3442400005261879

0.7789752992375562%Pruned 655
Average concept difference :  54.58333333333332
Average alignment:  0.29560177197214216
    
    
bert cofi c->c (at earlier iters concrpt dif is 10% lower & alignment is 6% higher. at 58 it shifts to more dif behavior)
so initially: sample i wrongly classif because it placed emphasus on a suboptimal neuron (ie pruning made the neuron less optimal for tha sample or removed kpwledge of the samole)
later: theres no dif numerically, neuron behavior changes a lot regrdless
    
0.25267370011269075%Pruned 7957
Average concept difference :  45.666666666666667
Average alignment:  0.452997426581569
   
0.4443280775114645%Pruned 7986
Average concept difference :  47.98333333333333
Average alignment:  0.42837691662833094
    
0.5826748728764346%Pruned 7869
Average concept difference :  53.01666666666667
Average alignment:  0.34265444518066945
    
0.6841850626856109%Pruned 7725
Average concept difference :  52.36666666666666
Average alignment:  0.3495491479942575
    
0.7789752992375562%Pruned 7653
Average concept difference :  51.8
Average alignment:  0.2954905626235995

tensor([ 0,  3,  6,  9, 12])

In [98]:
abs_map[2]

['cucumbers',
 'broccoli',
 'tasty',
 'pepper',
 'onion',
 'barney',
 'patrick',
 'mcdonalds',
 'avocados',
 'seafood',
 'strawberry',
 'lemons',
 'peppermints',
 'chili',
 'kfc',
 'banana',
 'ingredients',
 'nike',
 'joshua',
 'fruits',
 'nutritious',
 'tomatoes',
 'vegetable',
 'lemonade',
 'oranges',
 'fruit',
 'donald',
 'apple',
 'vegetables',
 'stanley',
 'coca',
 'soda',
 'cola',
 'blueberry',
 'fresh',
 'asparagus',
 'berries',
 'carrots',
 'cherry',
 'tomato',
 'hickock',
 'mcdonald',
 'foods',
 'juice',
 'carrot',
 'heineken',
 'ginger',
 'hershey',
 'delicious',
 'coconuts',
 'beverage',
 'lemon',
 'jeffs',
 'coconut',
 'apples',
 'john',
 'bananas',
 'chris',
 'maple',
 'grapes',
 'mary',
 'lime',
 'food',
 'pineapples',
 'freshly']

holisitc alignment

In [ ]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict, Counter
device = 'cuda' if torch.cuda.is_available() else 'cpu'
import json
import pandas as pd
import re
from collections import defaultdict
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import os
from itertools import combinations

def find_highest_activating_neurons(samples_activations, model_finallayerweights, d):
    flweights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    flweights = flweights.reshape((3, flweights.shape[0]//3))
    
    contribution = torch.tensor(samples_activations).abs().squeeze(0) * flweights
    
    total = contribution.sum(dim=1).argmax()  # best class row
    
    class_contributions = contribution[total]  # [1024]
    
    sorted_neurons = class_contributions.argsort(descending=True)  # indices sorted by contribution
    active_sorted_neurons = sorted_neurons[class_contributions[sorted_neurons] > 0]  # filter only active
    
    return active_sorted_neurons.tolist()
    

def find_highest_activating_neurons_at_cluster(cluster_mask, samples_activations,model_finallayerweights, d):
    num_activ = (samples_activations>0).sum()
   
    
    fl_cluster_weights = model_finallayerweights.detach().cpu().abs()[model_finallayerweights.detach().cpu().abs() > 0]
    
    fl_cluster_weights=fl_cluster_weights.reshape((3,fl_cluster_weights.shape[0]//3 ))
  
    contribution = torch.tensor(samples_activations).abs() * torch.tensor(cluster_mask) *fl_cluster_weights
    
    total = contribution.sum(dim=1).argmax()   # [1024]

    class_contributions = contribution[total]  # [1024]
    
    sorted_neurons = class_contributions.argsort(descending=True)  # indices sorted by contribution
    active_sorted_neurons = sorted_neurons[class_contributions[sorted_neurons] > 0]  # filter only active
    
    return active_sorted_neurons.tolist()

 

model='BERT'
method='lottery_ticket'
c=3

print(f"{model} {method} CLUSTER {c}")
path_to_experiment=os.path.join("/workspace/CCE_NLI",model, 'exp', method, 'Run0.25_5')
#dense_cw = pd.read_csv(os.path.join(path_to_experiment,  f'Prediction_CW_0_Pruning_Iter.csv')).set_index('Unnamed: 0').T
denseactivs = f"/workspace/CCE_NLI/{model}/activations/lottery_ticket/Run0.25_5/0_Pruning_Iter/final_layer_activations.pkl"
dense_expls_1,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster1IOUS1024N.csv'))
dense_expls_2,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster2IOUS1024N.csv'))
dense_expls_3,_=load_csv_data(os.path.join(f'/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5/Expls/', f'0.0%Pruned/Cluster3IOUS1024N.csv'))
with open(denseactivs, 'rb') as f:
    dense_activations = torch.tensor(pickle.load(f))
    
    
dense_path_root = f"/workspace/CCE_NLI/{model}/exp/lottery_ticket/Run0.25_5"
sparse_path_root = f"/workspace/CCE_NLI/{model}/exp/{method}/Run0.25_5"
dense_mask_1 = get_mask(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=1)
dense_mask_2 = get_mask(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=2)
dense_mask_3 = get_mask(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=3)

start=1
dense_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/lottery_ticket/Run0.25_5/0_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
dense_dead=get_subactiv(os.path.join(dense_path_root, 'Masks/0.0%Pruned'), cluster=c)
if '0.0%Pruned' in os.listdir(f"{sparse_path_root}/Expls"):
    start=1
    
for i, sparsity in enumerate(sorted(os.listdir(f"{path_to_experiment}/Expls"))):
   
    
    #if sparsity != '25.0%Pruned': continue
    if '.ipynb' in sparsity or '0.0%Pruned' in sparsity: continue
    #sparse_cw = pd.read_csv(os.path.join(path_to_experiment, f'Prediction_CW_{start}_Pruning_Iter.csv')).set_index('Unnamed: 0').T
    sparseactivs = f"/workspace/CCE_NLI/{model}/activations/{method}/Run0.25_5/{start}_Pruning_Iter/final_layer_activations.pkl"
    sparse_expls_1, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{1}IOUS1024N.csv'))
    sparse_expls_2, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{2}IOUS1024N.csv'))
    sparse_expls_3, _=load_csv_data(os.path.join(sparse_path_root, f'Expls/{sparsity}/Cluster{3}IOUS1024N.csv'))
    
    
    sparse_model_finallayerweights=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/model_best.pth', map_location=device)['state_dict']['mlp.3.weight']
    
    if method=='CoFi' and start>0:
        zs=torch.load(f'/workspace/CCE_NLI/{model}/models/{method}/Run0.25_5/{start}_Pruning_Iter/zs.pt', map_location=device)
        print(zs['final_mlp_hidden_z'].device, sparse_model_finallayerweights.device)
        sparse_model_finallayerweights = sparse_model_finallayerweights.mul(zs['final_mlp_hidden_z'].to(device))
    
    with open(sparseactivs, 'rb') as f:
        sparse_activations = torch.tensor(pickle.load(f))
        
    #c_2_w = all_correct_to_wrong(dense_cw, sparse_cw)
    
    average_alignment = defaultdict(list)
    alignment_between_neurons= []
    average_concept_diff = defaultdict(int)
    sparse_mask_1 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=1)
    sparse_mask_2 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=2)
    sparse_mask_3 = get_mask(os.path.join(sparse_path_root, 'Masks', sparsity), cluster=3)
  
    #sparse_dead = get_subactiv(os.path.join(dense_path_root, 'Masks', sparsity), cluster=c)
    start += 1
    #print(sparsity, len(c_2_w))
    died_neurons=0
    revived_neurons=0
    dense_highest_neuronsset=set()
    avg_cluster_holistic_alignment= 0
    concept_hol_dif=0
    for c2w_sample in range(600): #[:100]:
        sparse_highest_neurons1 = find_highest_activating_neurons_at_cluster(sparse_mask_1[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons2= find_highest_activating_neurons_at_cluster(sparse_mask_2[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
        sparse_highest_neurons3= find_highest_activating_neurons_at_cluster(sparse_mask_3[c2w_sample], sparse_activations[c2w_sample], sparse_model_finallayerweights, sparse_dead)
      
        dense_highest_neurons1 = find_highest_activating_neurons_at_cluster(dense_mask_1[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        dense_highest_neurons2 = find_highest_activating_neurons_at_cluster(dense_mask_2[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        dense_highest_neurons3 = find_highest_activating_neurons_at_cluster(dense_mask_3[c2w_sample], dense_activations[c2w_sample],dense_model_finallayerweights, dense_dead )
        
        difference_in_concepts=0
      
        #sparrse_expls_hol_covered =set().union(*[sparse_expls_1[n] for n in sparse_highest_neurons1])
        #sparrse_expls_hol_covered =set().union(*[sparse_expls_2[n] for n in sparse_highest_neurons2])
        sparrse_expls_hol_covered =set().union(*[sparse_expls_3[n] for n in sparse_highest_neurons3])
       
        #dense_expls_hol_covered = set().union(*[dense_expls_1[n] for n in dense_highest_neurons1])
        #dense_expls_hol_covered = set().union(*[dense_expls_2[n] for n in dense_highest_neurons2])
        dense_expls_hol_covered = set().union(*[dense_expls_3[n] for n in dense_highest_neurons3])
        
        
        concept_hol_dif += len(sparrse_expls_hol_covered & dense_expls_hol_covered)/len(dense_expls_hol_covered)
        #print(sorted(sparrse_expls_hol_covered & dense_expls_hol_covered))
        
        sparse_union_mask = sparse_mask_3.t()[sparse_highest_neurons3].any(dim=0)
        dense_union_mask = dense_mask_3.t()[dense_highest_neurons3].any(dim=0)
        
        avg_cluster_holistic_alignment  += activationsiou(sparse_union_mask,dense_union_mask )
        print("Cluster 3: ", iou(sparse_union_mask,dense_union_mask ))
        
    print("iou between sparse concept abstraction cverage and dense", sparsity, concept_hol_dif/600, avg_cluster_holistic_alignment/600)    

In [ ]:
llama lth
    alginment of cluste 1 for equiv neruons
    1
    1
    1
    1
    1
    1

    cluster 2:
    1
    1
    1
    1
    1

    how alignment are the cluster 3 masks for equiv neurons:
    0.7818
    0.7730
    0.7823
    0.7715
    0.7754
    
    
    
alignment of behavior of all neurons that fire at the respective cluster
bert lth
    how alignned are the cluster 1&2 masks for equiv neurons: 
    1
    1
    1
    1
    1
    
    how alignned are the cluster 3 masks for equiv neurons:
    0.8135
    0.8013
    0.8054
    0.8027
    0.8013
    
bert wanda:
    how alignned are the cluster 1&2 masks for equiv neurons: 
    1
    1
    1
    1
    1
    
    how alignned are the cluster 3 masks for equiv neurons: 
    0.8928
    0.8160
    0.7956
    0.8351
    0.8373
    
bert cofi
    how alignned are the cluster 1&2 masks for equiv neurons: 
    1
    1
    1
    1
    1
    
    how alignned are the cluster 3 masks for equiv neurons:
    0.8485
    0.8431
    0.8312
    0.8169
    0.7996

In [ ]:
bert wanda percnet sparse in dense with firing neurons t c3 holistically 
0.7158253470690867
0.5158081245464058
0.3367145398698728
0.3319527369198622
0.3217954856318257

bert wanda percnet sparse in dense with firing neurons t c3 holistically 
0.780325488503813
0.6618895856757624
0.49561810158466124
0.4297488691264148
0.387621722854215

bert lth percnet sparse in dense with firing neurons all clusters holistically  idnic concepst
0.7157827434841774
0.7150289015989116
0.705904915232019
0.7144492558370446
0.7104443351851716

bert wanda percnet sparse in dense with firing neurons holistically  (indiv concepts across all clusters)
0.7646508236874388
0.6824849638716762
0.5811244901600291
0.5204990223352913
0.49750395413909293

bert wanda percent groupings preserverd across all cluster:
25.0%Pruned 0.47863799756537856
43.75%Pruned 0.3298689137169994
57.812%Pruned 0.17422260587626442
68.359%Pruned 0.1283690198004774
76.27%Pruned 0.11109980672192672

so holistocally its not learning the same combos either at same cluster or acros clusters
its learning the same concepts but those are also in the untrained so its not indicative that the same concepst translate to same meaning

holsitcially cofi,lth is relearning the same absrtactions but it has that alignment even with pretrained or untrained (80-90%) 
    so its not indicative that the same concepst translate to same meaning

holsitically lth is also learning different groupings (35%avg fro lth with dense but 8%/ 12->14% w/ pretrained/untrained) 
holsitically cofi is also learning different groupings (38->33% fro cofi with dense but 8%/ 12->14% w/ pretrained/untrained) 
even wanda learns different grouppings which become more different with more prunign



basicaclly same indic concepts, diff groupings, same abstractions but same accuracy in lth/cofi
llama preserved along all firing neurons
    LTH groups
        0.30236508537915024
        0.2791821486572037
        0.2723068843244222
        0.27068421167371604
        0..


bert preserved along all firing neurons
    LTH indiv concepts 
        0.6979689263479909
        0.6975141678956701
        0.6900088414758995
        0.6972155524542846
        0.6933007260580186
    LTH groups
        0.3259912533643099
        0.3135241959204562
        0.31154873128645416
        0.30749358750641015
        0.31431311428325825
    LTH absracaiton
        0.9492857142857122
        0.9716666666666656
        0.9478571428571403
        0.9771428571428566
        0.9511904761904737
    
    COFI indiv concepts
        0.7400342995328111
        0.740564712147953
        0.7202402188955461
        0.7116983488704783
        0.6932507743365783
    COFI groiups
        0.3916439404098978
        0.391764689701342
        0.38044205211166937
        0.3555609300508366
        0.3396628598863317
    COFI abstacints
        0.9628571428571405
        0.9621428571428547
        0.9599999999999977
        0.9519047619047591
        0.9480952380952349
                
    WANDA indiv concets
        0.7589840355837185
        0.6803471127028979
        0.5770340951137035
        0.5061291564181638
        0.48619111751925287
    WANDA grouos
        0.45621053550492385
        0.3171332028348171
        0.17270628580981573
        0.1302600391771703
        0.11701319797904672
    WANDA abstracion
        0.9502380952380928
        0.9792857142857133
        0.9140873015872991
        0.848412698412695
        0.893531746031742
        
all preserves concepts
    - cofi does most, lth, then wanda (onlu low spasrsities)
none preserves groups
all preserves abstactions


all: preserves concepts and abstractions not goups  (Wanda only preserves concepts at low sparsities and higher sparsities become mroe different)
cofi: better job of preserving concepts and groups than lth and takes less space